This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
context = gx.get_context()
import logging

In [2]:
logging.basicConfig(level=logging.INFO, force = True)

In [3]:
## THIS IS REQUIRED FOR THE TECHNICAL VIEW HACK
# TODO handling of credentials not ideal, required for technical view fix
import os

from google.cloud import bigquery
import pandas as pd

sa_credentials_path=os.environ['SA_CREDENTIALS_PATH']
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_credentials_path
client = bigquery.Client()

In [4]:
gx_temp_schema = "tech_great_expectations_temp_ttl_7d"
date_filter = "hour <= '2099-12-31'"

In [5]:
def partition_enforcement_enabled(client, schema_name: str, table_name: str):
    partition_enforcement_enabled_sql = f"""
  SELECT
    option_value
  FROM
    {schema_name}.INFORMATION_SCHEMA.TABLE_OPTIONS
  WHERE
    table_name = '{table_name}'
  AND 
    option_name = 'require_partition_filter';
    """
    
    # TODO CHO20230622 handle exceptions
    partition_enforcement_enabled_res = pd.read_gbq(
        partition_enforcement_enabled_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )

    # TODO CHO20230622 handle multiple rows returned
    return(partition_enforcement_enabled_res.shape[0] > 0)

In [6]:
def get_partition_cols(client, schema_name: str, table_name: str):
    get_partition_cols_sql = f"""
  SELECT
    column_name,
    data_type,
    is_hidden
  FROM
    {schema_name}.INFORMATION_SCHEMA.COLUMNS
  WHERE
    table_name = '{table_name}'
  AND
    is_partitioning_column = 'YES';
    """
    
    # TODO CHO20230622 handle exceptions
    get_partition_cols_res = pd.read_gbq(
        get_partition_cols_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )

    # TODO CHO20230622 handle multiple rows returned
    return(get_partition_cols_res)

In [8]:
def create_or_replace_tech_view(
    client, 
    dataset_name: str, 
    table_name: str, 
    gx_temp_schema: str, 
    hidden_partition_col_sql: str = None,
    date_filter_sql: str = None
):

    # TODO CHO20230622 validate parameters, e.g. date filter
    # TODO CHO20230622 check first whether view exists before replacing
    fq_view_name = f"{gx_temp_schema}.v_unfiltered_{dataset_name}_{table_name}"
    query = f"""
    CREATE OR REPLACE VIEW `{fq_view_name}` AS (
    SELECT *{hidden_partition_col_sql} FROM {dataset_name}.{table_name} {date_filter_sql}
    )
    """

    # TODO CHO20230705 handle exceptions better
    resp=client.query(query)
    print(resp.result())
    return(fq_view_name)    

In [9]:
connection_string = f"""bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path={sa_credentials_path}"""

In [10]:
import yaml

In [11]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [12]:
datasource_config.get("project")

'gfw-google-827'

In [13]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

In [ ]:
gx_datasource.delete_asset

In [20]:
def add_datasource(client, datasource_name, dataset_name, table_name, version_number, gx_temp_schema, gx_datasource, default_max_date: str = "2099-12-31"):
    # TODO CHO20230705 handle non-date partition cols
    date_partition_cols = get_partition_cols(client, dataset_name, table_name).query("data_type.isin(['TIMESTAMP', 'DATE'])")
    partition_filter_enforced = partition_enforcement_enabled(client, dataset_name, table_name)

    hidden_partition_col_sql=''
    date_filter_sql=''
    used_partition_col=None
    date_partition_type=None
    
    if not partition_filter_enforced:
        logging.info(f"Partition enforcement not enabled, not applying date filter")
    else:
        if not date_partition_cols.shape[0]:
            logging.warn(f"""
                Partition enforcement enabled but no date columns found among partitions columns {partition_date_cols['column_name']}.
                Proceeding but eventually queries will fail!
            """)
        else:
            if not date_partition_cols.query("is_hidden=='YES'").shape[0]:
                source_partition_col = date_partition_cols["column_name"][0]
                used_partition_col=source_partition_col
            else:
                source_partition_col = date_partition_cols.query("is_hidden=='YES'")["column_name"][0]
                used_partition_col=source_partition_col.lstrip("_")
                hidden_partition_col_sql = f', {source_partition_col} AS {used_partition_col}'
                logging.info(f"""
                    Partition column {source_partition_col} is hidden and is therefore added to `SELECT *` statement
                """)

            # TODO CHO20230705 this is not 100% robust
            date_partition_type=date_partition_cols["data_type"][0]
            date_filter_sql = f"WHERE {source_partition_col} < '{default_max_date}'"
            logging.info(f"""
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: {date_filter}
            """)

    gx_view_name = create_or_replace_tech_view(
        client=client, 
        dataset_name=dataset_name, 
        table_name=table_name, 
        gx_temp_schema=gx_temp_schema, 
        hidden_partition_col_sql=hidden_partition_col_sql, 
        date_filter_sql=date_filter_sql
    )

    batch_metadata={
        'datasource_name': datasource_name, 
        'dataset_name': dataset_name, 
        'table_name': table_name, 
        'version_number': version_number
    }

    if date_partition_type:
        batch_metadata.update({'splitter_date_type': date_partition_type})
    
    asset_name=f"{datasource_name}-{version_number}"
    if asset_name in gx_datasource.get_asset_names():
        logging.info(f"{asset_name} already exists. Removing it before adding again")
        gx_datasource.delete_asset(asset_name)
        
    table_asset = gx_datasource.add_query_asset(
        name=asset_name,
        query=f"SELECT * FROM {gx_view_name}",
        batch_metadata=batch_metadata
    )

    # add datetime splitter and sorter if datetime partition column exists
    if used_partition_col:
        table_asset.add_splitter_column_value(used_partition_col)
        table_asset.add_sorters([f"-{used_partition_col}"])

In [21]:
for current_datasource in datasource_config.get("datasources"):
    datasource_name = current_datasource.get('name')
    if datasource_name not in gx_datasource.get_asset_names():
        print(f"Adding datasource '{current_datasource.get('name')}'")
        for current_version_number in current_datasource.get('versions'):
            print(f"Adding version '{current_version_number}'")
            config_current_version = current_datasource.get('versions').get(current_version_number)
            dataset_name = config_current_version.get('dataset')
            table_name = config_current_version.get('table')
            add_datasource(client, datasource_name, dataset_name, table_name, current_version_number, gx_temp_schema, gx_datasource)
    else:
        print(f"{datasource_name} already exists, skipping!")

Adding datasource 'messages'
Adding version '3.0.0'


INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: hour <= '2099-12-31'
            
INFO:root:messages-3.0.0 already exists. Removing it before adding again
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Adding version '2.5'


INFO:root:
                    Partition column _PARTITIONTIME is hidden and is therefore added to `SELECT *` statement
                
INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: hour <= '2099-12-31'
            
INFO:root:messages-2.5 already exists. Removing it before adding again
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Adding datasource 'satellite_timing_offsets'
Adding version '3.0.0'


...

INFO:root:Partition enforcement not enabled, not applying date filter
INFO:root:segment_info-2.5 already exists. Removing it before adding again
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Adding datasource 'segment_vessel'
Adding version '3.0.0'


INFO:root:Partition enforcement not enabled, not applying date filter
INFO:root:segment_vessel-3.0.0 already exists. Removing it before adding again
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


Adding version '2.5'


INFO:root:Partition enforcement not enabled, not applying date filter
INFO:root:segment_vessel-2.5 already exists. Removing it before adding again
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


In [93]:
gx_datasource.get_asset_names()

{'messages-2.5',
 'messages-3.0.0',
 'messages_positions-2.5',
 'messages_positions-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}